
# Proyecto: Naive Bayes con Estimación KDE para Mantenimiento Predictivo

Este notebook implementa un clasificador **Naive Bayes** que, en lugar de asumir una distribución Gaussiana para las características, **estima la densidad univariante de cada característica por clase utilizando KDE (Kernel Density Estimation)**. Se compara el rendimiento con `GaussianNB` de `sklearn`.

**Dataset:** AI4I 2020 Predictive Maintenance (archivo `ai4i2020.csv` subido).
**Columna objetivo:** `Machine failure` (0 = no falla, 1 = falla).


In [ ]:
# Cargar librerías y datos
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KernelDensity

csv_path = r"/mnt/data/ai4i2020.csv"
df = pd.read_csv(csv_path)
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))

In [ ]:
# Preparación de datos (seleccionar variables numéricas y eliminar identificadores)
target_col = "Machine failure"
X = df.select_dtypes(include=[np.number]).copy()
if target_col in X.columns:
    X = X.drop(columns=[target_col])
for drop_cand in ['UDI', 'Product ID', 'Product_ID', 'product_id']:
    if drop_cand in X.columns:
        X = X.drop(columns=[drop_cand])
y = df[target_col].astype(int)
X = X.fillna(X.mean())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print("Train/test sizes:", X_train.shape, X_test.shape)

In [ ]:
# Implementación del clasificador Naive Bayes con KDE
class NaiveBayesKDE:
    def __init__(self, bandwidth=None, kernel='gaussian'):
        self.bandwidth = bandwidth
        self.kernel = kernel
        self.kde_models = {}
        self.priors = {}
        self.features = None

    def _rule_of_thumb_bw(self, values):
        n = len(values)
        if n <= 1:
            return 1.0
        sigma = np.std(values, ddof=1)
        if sigma == 0 or np.isnan(sigma):
            return 1.0
        return 1.06 * sigma * (n ** (-1/5))

    def fit(self, X_df, y):
        self.features = list(X_df.columns)
        labels, counts = np.unique(y, return_counts=True)
        total = len(y)
        for lbl, cnt in zip(labels, counts):
            self.priors[lbl] = cnt / total
            self.kde_models[lbl] = {}
            X_lbl = X_df[y == lbl]
            for feat in self.features:
                values = np.asarray(X_lbl[feat]).reshape(-1,1)
                bw = self.bandwidth if self.bandwidth is not None else self._rule_of_thumb_bw(values.flatten())
                kde = KernelDensity(kernel=self.kernel, bandwidth=bw)
                if np.all(values == values[0]):
                    values = values + np.random.normal(scale=1e-6, size=values.shape)
                kde.fit(values)
                self.kde_models[lbl][feat] = kde

    def _log_likelihood_single(self, x_row, lbl):
        s = 0.0
        for i, feat in enumerate(self.features):
            kde = self.kde_models[lbl][feat]
            val = np.array(x_row[i]).reshape(1, -1)
            logd = kde.score_samples(val)[0]
            s += logd
        return s

    def predict(self, X_df):
        X_arr = X_df[self.features].values
        preds = []
        for row in X_arr:
            best_lbl = None
            best_score = -1e18
            for lbl in self.kde_models.keys():
                prior = np.log(self.priors.get(lbl, 1e-9))
                ll = self._log_likelihood_single(row, lbl)
                total_score = prior + ll
                if total_score > best_score:
                    best_score = total_score
                    best_lbl = lbl
            preds.append(best_lbl)
        return np.array(preds)

In [ ]:
# Entrenar y evaluar el modelo KDE-Naive Bayes
nb_kde = NaiveBayesKDE()
nb_kde.fit(X_train, y_train)
y_pred_kde = nb_kde.predict(X_test)
print("KDE-Naive Bayes Accuracy:", accuracy_score(y_test, y_pred_kde))
print("Classification report:\n", classification_report(y_test, y_pred_kde))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_kde))

# Comparación con GaussianNB
gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred_gnb = gnb.predict(X_test)
print("\nGaussianNB Accuracy:", accuracy_score(y_test, y_pred_gnb))
print("Classification report:\n", classification_report(y_test, y_pred_gnb))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_gnb))